# Introduction: Encoding Nominal Data

## What is Nominal Data?

**Nominal data** is a type of categorical data where categories have **no inherent order or ranking**. The categories are simply *labels* used to distinguish groups — there's no sense in which one label is "greater than" or "less than" another.

**Examples:** Colors (`Red`, `Blue`, `Green`), Cities (`Delhi`, `Mumbai`, `Pune`), Car Brands (`Audi`, `BMW`, `Porsche`)

This is different from **ordinal data**, where categories *do* have a natural order (e.g., `Low < Medium < High`, or `Bronze < Silver < Gold`). Ordinal data can often be encoded with simple integer mapping (0, 1, 2...) because the order carries meaning. For nominal data, assigning arbitrary integers (`Red=0, Blue=1, Green=2`) would falsely imply `Green > Red`, misleading any model that assumes numeric relationships (like distance-based or linear models).

## Why Do We Need Encoding at All?

Most machine learning algorithms are fundamentally mathematical — they operate on numbers, not text labels. Encoding is the process of converting categorical (string) data into a numerical representation that a model can consume, **without introducing artificial ordinal relationships** where none exist.

## What This Notebook Covers

This notebook implements — from scratch, without relying on library black-boxes — five progressively more sophisticated techniques for encoding nominal categorical data:

1. **One-Hot Encoding** — represent each category as a binary indicator vector
2. **Frequency / Count Encoding** — replace each category with how often it occurs
3. **Target / Mean Encoding** — replace each category with the average target value for that category
   - **3.1 Smoothing** — a regularized variant that reduces overfitting on rare categories
   - **3.2 Out-of-Fold Encoding** — a leakage-safe variant using cross-validation folds

Each technique makes a different trade-off between **information richness**, **dimensionality**, **overfitting risk**, and **target leakage** — which is exactly what we'll unpack, one section at a time.

---

# 1. One-Hot Encoding — Theory

## What It Is

One-Hot Encoding converts a categorical column with *k* unique categories into *k* (or *k−1*) new binary columns. For each row, the column corresponding to its category gets a `1`, and all other columns get `0`.

**Example:** `['Red', 'Blue', 'Green']` becomes:

| Blue | Green | Red |
|------|-------|-----|
| 0    | 0     | 1   |
| 1    | 0     | 0   |
| 0    | 1     | 0   |

This is the most direct way to represent nominal data numerically: it introduces **zero ordinal relationship** between categories — no category is treated as "bigger" or "closer" to another. Every category sits at equal Euclidean distance from every other in the encoded space.

## The `drop_first` Parameter

Encoding `k` categories technically only requires `k−1` binary columns — the "all zeros" row implicitly represents the dropped category (this is exactly what the notebook's `drop_first=True` path does). This matters specifically for **linear models** (Linear/Logistic Regression): if you keep all `k` columns, they become perfectly collinear (they always sum to 1), which causes the **dummy variable trap** — the design matrix becomes singular, and coefficient estimates become unstable or undefined.

- **Tree-based models** (Random Forest, XGBoost, Decision Trees): collinearity doesn't break anything — you can safely keep `drop_first=False`.
- **Linear/Logistic Regression, SVM, Neural Nets with regularization**: prefer `drop_first=True` to avoid the collinearity issue.



## Implementation 

In [1]:
import numpy as np

# Creating New unique Categories for given data
def OneHot_Unique_Categories(data, drop_first=False):
    
    """
    1. This function finds unique columns and fixes order for the given data
    2. If drop_first = True, then the first category is dropped from encoding
        (it's "all zeros" rows implicitly represents it )
    """
    
    unique_values = sorted(set(data))
    if drop_first:
        unique_values.pop(0)
        
    return unique_values    


# Creating One-Hot Encoding for given data
def OneHot_Encoder(data, categories, drop_first=False, all_categories=None):
    
    """
    1. This function transforms the given data into one-hot encoding
    2. If drop_first = True, then the first category is dropped from encoding
        (it's "all zeros" rows implicitly represents it )
    """
    
    encoded = []
    for val in data:
        row = [0]*len(categories)
        if val in categories:
            index = categories.index(val)
            row[index] = 1
        else:
            # val must be the dropped first category (all zero row)
            if drop_first and val==all_categories[0]:
                pass # row stays all zero
            else:
                raise ValueError(f"Value {val} not found in categories")
        encoded.append(row)
        
    return np.array(encoded)


# Decoding One-Hot Encoding for given data
def OneHot_Decoder(encoded_data, categories, all_categories):
    
    """ 
    1. This function takes the encoded data and returns the original categories
    2. It assumes that the encoded data is one-hot encoded and that all categories are present 
    
    """
    
    decode=[]
    for val in encoded_data:
        for idx in range(len(val)):
            if val[idx]==1:
                decode.append(categories[idx])
        if sum(val)==0:
            decode.append(all_categories[0])        
    
    return decode            




#=================================================================================================
# Dummy Example 1: drop_first=False
print("-----"*10)
print("When drop_first = False")

# Given data
data = ['Red','Blue','Green','Red','Green']

# All Unique Categories
all_categories = sorted(set(data))
print("All Unique Categories: ",all_categories)

# Categories without dropping first category
categories = OneHot_Unique_Categories(data)
print("Unique Categories(drop_first=False): ",categories,"\n")

# One-Hot Encoding
encoded_data = OneHot_Encoder(data, categories)
print("One-Hot Encoded Data:\n ",encoded_data,"\n")

# Decoding
decoded_data = OneHot_Decoder(encoded_data, categories, all_categories)
print("Given Data: ",data)
print("Decoded Data: ",decoded_data)


# ===============================================================================================
# Dummy Example 2: drop_first=True
print("-------"*10)
print("When drop_first = True")

# Given data
data = ['Audi', 'BMW', 'Porche', 'Lamborguini', 'Maclan', 'Audi', 'Lamborguini', 'BMW', 'BMW']

# All Unique Categories
all_categories = sorted(set(data))
print("All Unique Categories: ",all_categories)

# Categories with dropping first category
categories = OneHot_Unique_Categories(data, drop_first=True)
print("Unique Categories(drop_first=True): ",categories,"\n")

# One-Hot Encoding
encoded_data = OneHot_Encoder(data, categories, drop_first=True, all_categories=all_categories)
print("One-Hot Encoded Data:\n ",encoded_data,"\n")

# Decoding
decoded_data = OneHot_Decoder(encoded_data, categories, all_categories)
print("Given Data: ",data)
print("Decoded Data: ",decoded_data)

--------------------------------------------------
When drop_first = False
All Unique Categories:  ['Blue', 'Green', 'Red']
Unique Categories(drop_first=False):  ['Blue', 'Green', 'Red'] 

One-Hot Encoded Data:
  [[0 0 1]
 [1 0 0]
 [0 1 0]
 [0 0 1]
 [0 1 0]] 

Given Data:  ['Red', 'Blue', 'Green', 'Red', 'Green']
Decoded Data:  ['Red', 'Blue', 'Green', 'Red', 'Green']
----------------------------------------------------------------------
When drop_first = True
All Unique Categories:  ['Audi', 'BMW', 'Lamborguini', 'Maclan', 'Porche']
Unique Categories(drop_first=True):  ['BMW', 'Lamborguini', 'Maclan', 'Porche'] 

One-Hot Encoded Data:
  [[0 0 0 0]
 [1 0 0 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]
 [0 0 0 0]
 [0 1 0 0]
 [1 0 0 0]
 [1 0 0 0]] 

Given Data:  ['Audi', 'BMW', 'Porche', 'Lamborguini', 'Maclan', 'Audi', 'Lamborguini', 'BMW', 'BMW']
Decoded Data:  ['Audi', 'BMW', 'Porche', 'Lamborguini', 'Maclan', 'Audi', 'Lamborguini', 'BMW', 'BMW']



## Pros

1. **No false ordinal relationship** — mathematically the safest representation for true nominal data.

2. **Works well with linear models** — each category gets its own independent, interpretable coefficient.

3. **Simple, deterministic, no information leakage** — doesn't depend on the target variable at all, so it's safe to fit before any train/test split concerns around leakage (though you should still fit only on train data to keep unseen categories consistent).

4. **Widely supported** — every ML library and framework handles one-hot encoded input natively.

## Cons

1. **Curse of dimensionality**: a column with 1,000 unique categories (e.g., zip codes, user IDs) creates 1,000 new sparse columns — this blows up memory, training time, and can hurt models sensitive to high dimensionality (like KNN, distance-based clustering).

2. **Sparsity**: most values in the encoded matrix are 0, which is inefficient unless you use sparse matrix representations.
3. **Memory usage**: each category requires its own column, which can lead to large memory foot prints.

4. **No relationship information captured**: it treats "Delhi" and "Mumbai" as equally unrelated as "Delhi" and "Pune" — even if there's useful signal in how frequent or how predictive a category is (this is exactly the gap Frequency and Target Encoding later address).

5. **Doesn't generalize to unseen categories** at inference time without special handling.

6. **Not ideal for tree-based models with high-cardinality features** — one-hot encoding can make trees inefficient because a single feature's signal gets fragmented across many binary columns, making it harder for the tree to find good splits (this is a known weakness discussed extensively in the CatBoost/LightGBM literature).

## When to Avoid It

1. **High-cardinality nominal features** (hundreds or thousands of unique categories) — zip codes, user IDs, product SKUs, free-text categories. Use Frequency or Target Encoding instead.

2. **Tree-based models with many categories** — trees generally handle raw categorical splits or target/frequency encodings more efficiently than heavily fragmented one-hot columns.

3.  **Memory-constrained pipelines** — if the dataset is large and cardinality is even moderate (50+), the sparse blow-up can become a real bottleneck.

4. **When you need the encoding to carry predictive signal** (e.g., "this category tends to correlate with the target") — one-hot is target-agnostic by design, so it can't help here. Target Encoding is built for this.

---

# 2. Frequency / Count Encoding — Theory

## What It Is

Frequency (or Count) Encoding replaces each category with either:
- **Raw count**: the number of times that category appears in the dataset, or
- **Normalized frequency**: that count divided by the total number of rows (i.e., a proportion between 0 and 1).

**Example:** `['Red','Blue','Green','Red','Green','Blue','Red']`
- Raw counts: `Red → 3, Blue → 2, Green → 2`
- Normalized: `Red → 0.429, Blue → 0.286, Green → 0.286`

Unlike One-Hot Encoding, this collapses each category into a **single numeric column** rather than expanding it into many — the categorical column stays one-dimensional.




## Implementation

In [2]:

# Frequency Count Encoding Function
def fit_count_encoder(data, normalize=False):
    
    """ 
    This Function takes a list of data and returns the frequency count encoding for each category.
    It also optionally normalizes the counts to ensure that all categories have
    """
    counts = {}
    
    for val in data:
        if val in counts:
            counts[val] += 1
        else:
            counts[val] = 1
        
    if normalize:
        total = len(data)
        for category in counts:
            counts[category] = counts[category] / total
    
    
    return counts

# Encoding Function
def count_encoder(data, mapping):
    """
    This Function takes a list of data and a mapping (output from fit_count_encoder), and returns the transformed data.   
    """
    encoded = []
    
    for val in data:
        encoded.append(mapping[val])
    return encoded    


# ====================================================================
# Dummy Example: 1 ( Raw counts)
data = ['Red','Blue','Green','Red','Green','Blue','Red']

mapping = fit_count_encoder(data, normalize=False)
print(mapping)

encoded_data = count_encoder(data, mapping)     
print(encoded_data)


# =====================================================================
# Dummy Example: 2 (Normalized Counts)
data = ['Red','Blue','Green','Red','Green','Blue','Red']

mapping = fit_count_encoder(data, normalize=True)
print(mapping)


encoded_data = count_encoder(data, mapping)     
print(f"Encoded Data: {np.round(encoded_data, decimals=3)}")

{'Red': 3, 'Blue': 2, 'Green': 2}
[3, 2, 2, 3, 2, 2, 3]
{'Red': 0.42857142857142855, 'Blue': 0.2857142857142857, 'Green': 0.2857142857142857}
Encoded Data: [0.429 0.286 0.286 0.429 0.286 0.286 0.429]


## Pros

1.  **Extremely low dimensionality** — one column, regardless of how many categories exist. This makes it a strong choice for high-cardinality features where One-Hot would explode.

2.  **Compact and memory-efficient** — no sparse matrices, no blown-up feature space.

3.  **Captures a genuinely useful signal in many cases** — rare vs. common categories often behave differently, and this encoding surfaces that directly.

4.  **Works well with tree-based models** — trees can split cleanly on frequency thresholds (e.g., "categories with count > 50 behave differently").

5.  **No target leakage** — like One-Hot, it doesn't use the target variable at all, so it's safe with respect to label leakage (though you should still compute counts only on the training set to avoid train/test contamination).

## Cons

1. **Collision problem**: two *completely different* categories that happen to occur the same number of times get the exact same encoded value, even if they have nothing else in common. The model has no way to distinguish them anymore — information is genuinely lost.

2. **No ordinal meaning is actually justified**: even though the output is numeric, "count" doesn't necessarily have a meaningful linear relationship with the target unless frequency itself matters to the problem. Linear models may pick up spurious relationships here.

3. **Unstable across train/test splits or over time**: if the category distribution shifts (e.g., a rare category becomes common after deployment, or new data arrives), the encoding becomes stale and needs to be refit.

4. **Doesn't work for unseen categories** without a fallback strategy.

5. **Not directly informative about the target** — it tells you about the category's popularity, not its relationship to what you're trying to predict (that gap is what Target Encoding addresses next).

## When to Avoid It

1. **When many categories share the same frequency** — the collision problem becomes severe, and the model effectively loses the ability to distinguish between them.

2. **When the *relationship to the target* matters more than popularity** — e.g., predicting churn where a rare category might be highly predictive; frequency alone won't surface that signal. Target Encoding is better suited here.

3. **Non-stationary data** (frequencies drift significantly over time, e.g., trending topics, seasonal categories) — the encoding needs frequent refitting or it becomes misleading.

4. **When category counts are themselves sensitive/leaky information** — e.g., if "count" indirectly reveals something about a held-out period or future data, this can silently introduce leakage.

# 3. Target / Mean Encoding — Theory

## What It Is

Target Encoding (also called Mean Encoding) replaces each category with the **average value of the target variable** for that category. Unlike One-Hot and Frequency Encoding, this is a **supervised** encoding technique — it directly uses the label/target during encoding, not just the feature values.

**Example:** Predicting purchase (1) vs. no purchase (0), by city:

`data = ['Delhi','Mumbai','Delhi','Delhi','Mumbai','Delhi','Delhi','Pune']`
`target = [1, 1, 1, 0, 0, 1, 1, 1]`

- Delhi → mean of [1,1,0,1,1] = 0.8
- Mumbai → mean of [1,0] = 0.5
- Pune → mean of [1] = 1.0

Each city is now represented by a single number that directly encodes "how likely is the target to be 1, given this category" — which is often exactly the kind of signal a model wants.

## Why It Works

This is the most **information-dense** of the three techniques so far, because it doesn't just describe the category (like frequency does) — it describes the category's **relationship with what you're trying to predict**. For high-cardinality nominal features, this can dramatically outperform One-Hot or Frequency encoding because it compresses potentially hundreds of categories into a single, highly predictive numeric column.





# 3. Target/Mean Encoding

In [3]:


def fit_target_encoder(data, target):
    """ Data: List of categorical data
    target: List of binary targets (0 or 1) 
    Returns a dictionary mapping each category to its mean target value """
    
    sums={}
    counts={}
    
    for val,y in zip(data, target):
        if val in sums:
            sums[val] +=y
            counts[val] +=1
        else:
            sums[val]=y
            counts[val]=1
            
    mapping={}
    
    for category in sums:
        mapping[category] = sums[category]/counts[category]
        
    return mapping    


def transform_target(data, mapping):
    
    """ 
    This function takes a list of categorical data and a mapping dictionary
    and returns the encoded data as a list of floats 
    
    """
    encoded=[]
    
    for val in data:
        encoded.append(mapping[val])
    
    return encoded    


# Dummy Example: Predicting whether someone buys(1) or not (0), by city

data = ['Delhi', 'Mumbai', 'Delhi', 'Delhi', 'Mumbai', 'Delhi', 'Delhi', 'Pune']
target = [1, 1, 1, 0, 0, 1, 1, 1]

# Mapping the categorical data to numerical values
mapping = fit_target_encoder(data, target)
print("Mapping : ",mapping)

# Encoding the target variable using the fitted encoder
encoded_data = transform_target(data, mapping)
print(f"Given data : {data}")
print(f"Encoded data : {encoded_data}")

Mapping :  {'Delhi': 0.8, 'Mumbai': 0.5, 'Pune': 1.0}
Given data : ['Delhi', 'Mumbai', 'Delhi', 'Delhi', 'Mumbai', 'Delhi', 'Delhi', 'Pune']
Encoded data : [0.8, 0.5, 0.8, 0.8, 0.5, 0.8, 0.8, 1.0]


## Pros

1. **Highly predictive** — directly encodes the empirical relationship between category and target, often the single most informative encoding for high-cardinality categorical features.

2. **Low dimensionality** — one column, same benefit as Frequency Encoding, but with actual predictive signal baked in rather than just popularity.

3. **Especially effective for high-cardinality features** in gradient boosting models (XGBoost, LightGBM, CatBoost), where it's a standard technique in competitive ML pipelines.

## Cons 

1. **Target Leakage**: this is the single biggest risk with target encoding. Because each row's encoding is computed using the *target values of other rows that share its category* — and in the naive version, **including its own target value** — the encoding can leak information about the label directly into the feature. A model can essentially "cheat" by learning to reverse-engineer the target from the encoding, leading to inflated training/validation performance that collapses on true unseen data.

2.  **Overfitting on rare categories**: a category that appears only once (like "Pune" with mean = 1.0 above) gets an extreme, unreliable estimate — its "mean" is based on a single data point and doesn't generalize. This is a classic small-sample-size problem.

3. **Only works for one target definition**: unlike One-Hot or Frequency Encoding, which are target-agnostic and reusable across problems, this encoding is tied to a *specific* target variable — you need to refit if the target changes.

4. **Requires care around train/test split**: the mapping must be fit *only* on training data; if computed on the full dataset (including test/validation), it directly leaks test-set label information into the features.

5. **Not naturally interpretable** in the same way One-Hot is — the encoded value is a statistical artifact of the data, not a simple category flag.

## When to Avoid It (Or Use With Extreme Caution)

1. **Never use the naive/raw version (as shown in this section) in production or serious modeling** without addressing leakage — it's a well-known way to accidentally build a model that appears very accurate but fails on real unseen data. This is exactly why the notebook introduces **Smoothing (3.1)** and **Out-of-Fold Encoding (3.2)** next — they exist specifically to fix these two problems.

2. **Small datasets with many rare categories** — the raw mean per category becomes noisy and unreliable without regularization (smoothing).

3. **When you can't guarantee strict train/test separation** in your pipeline — the risk of accidentally leaking target information is high, especially in cross-validation setups if not done carefully.

4. **Regression or multi-class targets need adapted versions** — the raw implementation here assumes a binary target; for regression, you'd average a continuous target, and for multi-class, you typically need one encoding per class (or a different technique altogether).


---

# 3.1 Smoothing (Regularized Target Encoding) — Theory

Smoothing fixes the **rare-category overfitting problem** from raw Target Encoding by blending each category's mean with the **global target mean**, weighted by how many samples that category has. Instead of trusting a category's own mean outright, the encoding "pulls" uncertain (low-sample) categories back toward the overall average.

The formula used :


$$\text{smoothed\_mean} = \frac{n \cdot \text{category\_mean} + \text{smoothing} \cdot \text{global\_mean}}{n + \text{smoothing}}$$



Where:
- `n` = number of samples in that category
- `category_mean` = raw target mean for that category
- `global_mean` = target mean across the entire dataset
- `smoothing` = a tunable hyperparameter controlling how much weight goes to the global mean

## Why It Works — The Intuition

This is a **weighted average** between two estimates:
1. The category's own observed mean (trustworthy when `n` is large)
2. The global mean (a safe fallback when `n` is small)

As `n → ∞` (lots of data for that category), the formula converges to `category_mean` — the global mean's influence becomes negligible. As `n → 0` (rare category), the formula converges to `global_mean` — you effectively ignore the unreliable per-category estimate. The `smoothing` parameter controls the "pivot point": a higher smoothing value means you need *more* samples before you start trusting the category's own mean over the global average.

This is conceptually the same idea behind **Bayesian shrinkage / empirical Bayes estimators** — you're blending a noisy local estimate with a stable prior (the global mean), and letting sample size determine how much you trust the local signal.




## Implementation

In [4]:

# Define a function to fit a target encoder with smoothing
def fit_target_encoder_smoothed(data, target, smoothing=1):
    
    """ 
    This function encode the data using a smoothed version of the target encoding.
    The smoothing parameter controls how much weight is given to the mean of the target
    
    Higher Smoothing -> More Weight on Mean (Pulls Data Closer to Global Mean)
    """
    # Calculate the global mean
    global_mean = sum(target)/ len(target)
    
    sums = {}
    counts = {}
    
    # Iterate over the data and calculate the sum of each value and its count
    for val, y in zip(data, target):
        if val in sums:
            sums[val] += y
            counts[val] += 1
        else:
            sums[val]=y
            counts[val]=1
    
    # Calculate the smoothed mean for each value
    mapping={}
    
    for category in sums:
        n = counts[category]
        category_mean = sums[category]/n
        
        # weighted blend : More weight to category_mean as n grows
        smoothed_value = (category_mean * n + global_mean * smoothing) / (n + smoothing)
        mapping[category] = smoothed_value
    
    return mapping    


# Transform Target with Smoothing
def transform_target_smoothing(data, mapping):
    encoded = []
    
    for val in data:
        encoded.append(mapping[val])
    return encoded


# ===================================================================================
# Dummy Example
data = ['Delhi', 'Mumbai', 'Delhi', 'Delhi', 'Mumbai', 'Delhi', 'Delhi', 'Pune']
target = [1, 1, 1, 0, 0, 1, 1, 1]

mapping_smoothing = fit_target_encoder_smoothed(data, target, smoothing=50)
print(f"Mapping : {mapping_smoothing}")

encoded_data_smoothing = transform_target_smoothing(data, mapping_smoothing)
print(f"Encoded Data : {np.round(encoded_data_smoothing, decimals=4)}")

Mapping : {'Delhi': 0.7545454545454545, 'Mumbai': 0.7403846153846154, 'Pune': 0.7549019607843137}
Encoded Data : [0.7545 0.7404 0.7545 0.7545 0.7404 0.7545 0.7545 0.7549]


## Pros

1. **Directly solves the rare-category overfitting problem**: a category seen once no longer gets an extreme, unreliable value — it gets pulled toward a sensible default.

2. **Tunable trade-off**: `smoothing` gives you explicit, interpretable control over the bias-variance trade-off — low smoothing trusts the data more (higher variance, less bias), high smoothing trusts the global mean more (more bias, less variance).

3. **Still cheap to compute** — no change in computational complexity vs. raw target encoding, just an extra couple of arithmetic operations.

4. **Well-established technique** — this is essentially how CatBoost and other production-grade libraries implement target encoding internally (with some additional refinements).

## Cons

1. **Does NOT fix target leakage** — this is the critical point the notebook itself calls out. Even with smoothing, each row's encoding still (partially) includes its own target value in the category's mean computation. A row still gets to "see" a small piece of its own label reflected back in its feature — smoothing only tempers this effect for *rare* categories, it doesn't eliminate it for common ones.

2. **Introduces a new hyperparameter to tune** (`smoothing`) — this adds complexity to the modeling pipeline; picking the wrong value under- or over-regularizes.

3. **The right smoothing value is dataset-dependent** — there's no universal default; it typically needs cross-validation or domain judgment to set well.

4. **Still tied to a specific target** — like raw target encoding, this must be refit if the target variable changes.

## When to Avoid It

1. **When you need a leakage-free encoding for a rigorous modeling pipeline** — smoothing alone is not sufficient; you need Out-of-Fold Encoding (Section 3.2) on top of it, especially for high-stakes or competition-grade models.

2. **When you don't have time/data to properly tune `smoothing`** — an arbitrary/default smoothing value may under- or over-correct, sometimes performing worse than either the raw mean or the global mean alone.

3. **Very small datasets overall** — if the entire dataset is small, even the "global mean" fallback is a noisy estimate, and no amount of smoothing can fully compensate for a fundamentally data-starved problem.

# 3.2 Out-of-Fold (OOF) Target Encoding — Theory



Out-of-Fold Encoding is the technique that finally **eliminates target leakage** from target encoding, which smoothing alone could not fully fix. The core idea: split the data into `n_folds` folds, and for each fold, compute the category means using **only the other folds** — never the fold being encoded. This guarantees that no row's encoded value is ever derived, even partially, from its own target label.

This is directly analogous to **k-fold cross-validation**, applied not to model evaluation but to feature engineering itself.

## Why It Works

The fundamental leakage problem in raw (and smoothed) target encoding is that a row's target value contributes to the mean used to encode that *same* row. OOF encoding breaks this dependency structurally:

1. Split the dataset into `n_folds` folds (e.g., 3 or 5).
2. For each fold `k`:
   - Compute category means using **all data except fold `k`** (the "training folds").
   - Use that mapping to encode **only the rows in fold `k`** (the "held-out fold").
3. Repeat for every fold, so every row eventually gets encoded — but always using means computed from data that excludes it.

The result: every row's encoded value is now a genuinely out-of-sample estimate of "how does this category typically relate to the target," with **zero direct contribution from that row's own label**. This mirrors how the encoding will behave at inference time on unseen data — which is the entire point.




## Implementation

In [5]:
def fit_transform_target_OOF(data, target, n_folds=3, smoothing=1):
    """
    This function fits a target encoder to the data and transforms it using out-of-fold encoding.
    Returns out-of-fold encoded values (same length as data)
    NOTE: In this method Each row's encoding is computed WITHOUT using that row's own target.
    
    """
    
    n = len(data)
    fold_size = n // n_folds
    fold_ids = []
    
    for i in range(n):
        fold_ids.append(min(i//fold_size, n_folds-1)) # Assign each row to a fold
        
    global_mean = sum(target) / n 
    encoded = [None] * n 
    
    # Split into "held-out fold" (being encoded) and "training folds" (used to compute means)
    
    for fold in range(n_folds):
        train_data, train_target = [], []
        
        for i in range(n):
            if fold_ids[i] != fold:
                train_data.append(data[i])
                train_target.append(target[i])
    
    # Compute smoothed mapping using ONLY the training folds
    sums, counts = {},{}
    
    for val, y in zip(train_data, train_target):
        if val in sums:
            sums[val] += y
            counts[val] += 1
        else:
            sums[val] = y
            counts[val] = 1
            
    mapping={}
    for category in sums:
        cn =counts[category]
        category_mean = sums[category]/cn
        mapping[category]= (category_mean * cn + global_mean * smoothing) / (cn + smoothing)

    # Encode the held-out fold using the computed mapping
    for i in range(n):
        if fold_ids[i] == fold:
            val = data[i]
        if val in mapping:
            encoded[i] = mapping[val]
        else:
            encoded[i] = global_mean # Use the global mean as a default value for unseen category 
    
    return encoded

# Dummy example (bigger, so folds make sense)
data =   ['Delhi','Mumbai','Delhi','Pune','Mumbai','Delhi','Pune','Mumbai','Delhi','Pune']
target = [1,       0,       1,      0,     1,       0,      0,     1,       1,      0]

encoded = fit_transform_target_OOF(data, target, n_folds=3, smoothing=10)

for city, val in zip(data, encoded):
    print(city, "->", round(val, 3))

Delhi -> 0.538
Mumbai -> 0.538
Delhi -> 0.538
Pune -> 0.538
Mumbai -> 0.538
Delhi -> 0.538
Pune -> 0.455
Mumbai -> 0.5
Delhi -> 0.538
Pune -> 0.455



## Pros

1. **Solves target leakage properly** — this is the only technique among the four that gives a mathematically sound guarantee that a row's own label never contributes to its own encoded feature value.

2. **Combines naturally with smoothing** — the notebook's implementation applies smoothing *within* each fold's training-only computation, so both problems (overfitting on rare categories *and* leakage) are addressed simultaneously.

3. **Realistic validation performance** — because the encoding process now mimics how it would behave on truly unseen data, cross-validated model performance becomes a trustworthy estimate of real-world performance, instead of being artificially inflated.

4. **Standard in serious/competitive ML pipelines** — this (or a close variant) is what's used under the hood in libraries like CatBoost's "ordered target statistics," and is considered best practice for target encoding in Kaggle-style competitions.

## Cons

1. **More complex to implement and reason about** — requires careful fold management, and bugs here are easy to introduce silently (e.g., accidentally using the wrong fold, or leaking through fold order in time-series data).

2. **Computationally more expensive** — you're essentially fitting `n_folds` separate encoders instead of one, though the overhead is usually minor for typical dataset sizes.

3. **Still doesn't guarantee zero leakage in poorly designed folds** — for example, if folds aren't shuffled and the data has some ordering/grouping structure (e.g., time-based data, or duplicate rows for the same entity across folds), leakage can still creep in through row-to-row correlation rather than through the direct target-mean computation.

4. **Choice of `n_folds` matters**: too few folds (e.g., 2) means each training subset is small and the fold-level means are noisier; too many folds means each held-out fold is tiny and encoding computation overhead increases, with diminishing leakage-reduction benefit beyond a point.

5. **Extra care needed for final deployment mapping** — the mapping used to encode genuinely new/unseen data at inference time should be fit on the *full* training set (not per-fold), which is a subtly different mapping than any of the fold-specific ones used during training — this distinction is easy to get wrong.

## When to Avoid It

1. **Very small datasets** — if the dataset is small to begin with, splitting into folds leaves each fold-training subset even smaller, which can make per-fold category means highly unstable, especially for rare categories, even with smoothing.

2. **When simplicity/speed matters more than rigor** — for a quick baseline model or exploratory analysis, the added complexity may not be justified; smoothed (non-OOF) target encoding might be an acceptable, faster compromise, accepting a small amount of leakage.

3. **Grouped or time-dependent data without adapting the fold strategy** — naive random fold splitting can still leak information if rows aren't independent (e.g., multiple rows belonging to the same customer, or time-series data where future information leaks into past folds). In such cases, use **GroupKFold** or **time-based/rolling-window splits** instead of plain random folds.

3. **When category cardinality is extremely high relative to dataset size** — with folds further splitting the data, sparse categories can end up entirely absent from some training folds, forcing frequent fallback to the global mean and diluting the technique's benefit.

# Summary: Choosing a Nominal Encoding Technique

## Quick Comparison Table

| Technique | Dimensionality | Uses Target? | Leakage Risk | Handles High Cardinality | Best Model Fit |
|---|---|---|---|---|---|
| **One-Hot** | High (k or k−1 columns) | No | None | Poor (dimensionality blow-up) | Linear/Logistic Regression, SVM, Neural Nets |
| **Frequency/Count** | Low (1 column) | No | None | Good | Tree-based models (RF, XGBoost) |
| **Target/Mean (raw)** | Low (1 column) | Yes | **High** | Good | Not recommended raw — leakage-prone |
| **Smoothed Target** | Low (1 column) | Yes | Still present (reduced for rare categories) | Good | Tree-based models, with caution |
| **Out-of-Fold Target** | Low (1 column) | Yes | **Eliminated** (by design) | Excellent | Tree-based models, competition-grade pipelines |

## Decision Guide

**Start by asking: how many unique categories does the feature have?**

- **Low cardinality (≤ ~15 categories)** → **One-Hot Encoding** is usually the safe default, especially with linear models. No leakage risk, no tuning needed, fully interpretable.

- **High cardinality (hundreds/thousands of categories), no target-relevant signal needed** → **Frequency/Count Encoding**. Cheap, dense, no leakage, decent for tree models.

- **High cardinality, target signal matters, and you're doing quick exploratory work** → **Smoothed Target Encoding**. Accept some residual leakage risk in exchange for simplicity — fine for baselining, not for final/production models.

- **High cardinality, target signal matters, and you need a trustworthy, deployable model** (competitions, production ML, anything where leaked validation scores would mislead you) → **Out-of-Fold Target Encoding**. The only option here with a real leakage guarantee.

## Model-Type Cheat Sheet

- **Linear models (Regression, Logistic Regression, SVM)** → One-Hot (with `drop_first=True` to avoid the dummy variable trap). Target encoding's continuous values can also work, but One-Hot is more standard and interpretable.
- **Distance-based models (KNN, K-Means)** → One-Hot for low cardinality; avoid raw target/frequency encoding since arbitrary numeric scale can distort distances.
- **Tree-based models (Random Forest, XGBoost, LightGBM, CatBoost)** → Frequency or (Smoothed/OOF) Target Encoding scale much better with cardinality than One-Hot, and trees exploit numeric thresholds well.
- **Neural Networks** → Often better served by **learned embeddings** (not covered in this notebook) for high-cardinality categoricals, though One-Hot works fine for low cardinality.

## The One Rule That Matters Most

**Never fit any encoder — Frequency, Target, or otherwise — on anything but the training split.** Fitting on full data (train+test/validation) before splitting is the single most common way leakage sneaks into a pipeline, regardless of which encoding technique you choose.